# SCRIPT 01: Multi-Sensor Harmonization and Time-Series Extraction
**Project:** Andean Fire Ecosystem Dynamics (2017-2025)

**Description:**
This script extracts monthly NDVI and NBR time-series data by combining Sentinel-2, Landsat 8, and Landsat 9 imagery. It utilizes direct Google Earth Engine Level-2 Surface Reflectance collections to minimize data loss over rugged topography.

A custom harmonization approach is applied, featuring strict scaling and clamping of indices specifically tailored to non-fire-adapted high-altitude ecosystems, ensuring a robust temporal baseline for vegetation analysis.

In [ ]:
"""
================================================================================
SCRIPT 01: Multi-Sensor Harmonization and Time-Series Extraction
Project: Andean Fire Ecosystem Dynamics (2017-2025)
Description: Extracts NDVI and NBR time-series data combining Sentinel-2,
Landsat 8, and Landsat 9 utilizing direct Google Earth Engine Level-2
Surface Reflectance collections.
================================================================================
"""

import ee
import geemap
import pandas as pd
from datetime import datetime

# ------------------------------------------------------------------------------
# 1. INITIALIZATION AND AUTHENTICATION
# ------------------------------------------------------------------------------
print("🔄 Initializing Google Earth Engine...")
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='my-project-123-477720') # Make sure to use your project ID
print("✅ GEE Initialized Successfully.")

# ------------------------------------------------------------------------------
# 2. STUDY AREA & PARAMETERS
# ------------------------------------------------------------------------------
# Replace with your actual asset path
ROI_PATH = 'projects/my-project-123-477720/assets/ACR_TITANKAYOCC_UNION'
roi = ee.FeatureCollection(ROI_PATH)

START_YEAR = 2017
END_YEAR = 2025
CLOUD_THRESHOLD = 70

print(f"📍 Study Area Loaded. Timeframe: {START_YEAR}-{END_YEAR}")

# ------------------------------------------------------------------------------
# 3. HARMONIZATION FUNCTIONS
# ------------------------------------------------------------------------------
def process_s2(image):
    """
    Processes Sentinel-2 SR images.
    Applies QA60 cloud mask and calculates NDVI & NBR.
    """
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))

    # Calculate Indices
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI').clamp(-1, 1).float()
    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR').clamp(-1, 1).float()

    return image.updateMask(mask).addBands([ndvi, nbr]).copyProperties(image, ['system:time_start'])

def process_landsat(image):
    """
    Processes Landsat 8/9 Level-2 SR images.
    Applies QA_PIXEL mask, applies Scale Factors, and calculates NDVI & NBR.
    """
    qa = image.select('QA_PIXEL')
    # Bit 3 is cloud, Bit 4 is cloud shadow
    cloudShadowBitMask = 1 << 4
    cloudsBitMask = 1 << 3
    mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0).And(qa.bitwiseAnd(cloudsBitMask).eq(0))

    # Apply standard Level-2 Scale Factors
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)

    # Extract Specific Bands
    red = opticalBands.select('SR_B4')
    nir = opticalBands.select('SR_B5')
    swir = opticalBands.select('SR_B7')

    # Calculate Indices
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI').clamp(-1, 1).float()
    nbr = nir.subtract(swir).divide(nir.add(swir)).rename('NBR').clamp(-1, 1).float()

    return image.updateMask(mask).addBands([ndvi, nbr]).copyProperties(image, ['system:time_start'])

# ------------------------------------------------------------------------------
# 4. TIME-SERIES EXTRACTION (MONTHLY LOOP)
# ------------------------------------------------------------------------------
print("🚀 Extracting Monthly Constellation Data (S2 + L8 + L9)...")

months = list(range(1, 13))
years = list(range(START_YEAR, END_YEAR + 1))
data_matrix = []

for year in years:
    for month in months:
        # Define date range for the current month
        start_date = f'{year}-{month:02d}-01'
        # Handle end of month (simple approach using ee.Date)
        end_date = ee.Date(start_date).advance(1, 'month').format('YYYY-MM-dd').getInfo()

        # 1. Sentinel-2
        s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi)
              .filterDate(start_date, end_date)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_THRESHOLD))
              .map(process_s2)
              .select(['NDVI', 'NBR']))

        # 2. Landsat 8
        l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
              .filterBounds(roi)
              .filterDate(start_date, end_date)
              .filter(ee.Filter.lt('CLOUD_COVER', CLOUD_THRESHOLD))
              .map(process_landsat)
              .select(['NDVI', 'NBR']))

        # 3. Landsat 9 (Available post-2021)
        l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
              .filterBounds(roi)
              .filterDate(start_date, end_date)
              .filter(ee.Filter.lt('CLOUD_COVER', CLOUD_THRESHOLD))
              .map(process_landsat)
              .select(['NDVI', 'NBR']))

        # Merge Collections
        combined = s2.merge(l8).merge(l9)

        # Check if data exists for this month
        size = combined.size().getInfo()
        if size > 0:
            # Create a monthly median composite to reduce outliers
            monthly_composite = combined.median().clip(roi)

            # Reduce region to get the mean index values for the whole ROI
            stats = monthly_composite.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=roi.geometry(),
                scale=30, # Base scale
                maxPixels=1e9
            )

            ndvi_val = stats.get('NDVI').getInfo()
            nbr_val = stats.get('NBR').getInfo()

            # Ensure valid numbers (discard None)
            if ndvi_val is not None and nbr_val is not None:
                data_matrix.append({
                    'Date': f'{year}-{month:02d}-15', # Mid-month representation
                    'NDVI': round(ndvi_val, 4),
                    'NBR': round(nbr_val, 4),
                    'Image_Count': size
                })
                print(f"  [+] {year}-{month:02d} | Images: {size:02d} | NBR: {nbr_val:.3f} | NDVI: {ndvi_val:.3f}")
        else:
            print(f"  [-] {year}-{month:02d} | No valid cloud-free data.")

# ------------------------------------------------------------------------------
# 5. EXPORT RESULTS TO CSV
# ------------------------------------------------------------------------------
print("💾 Saving data to CSV...")
df = pd.DataFrame(data_matrix)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by='Date')

export_filename = '1_Global_Monthly_Stats.csv'
df.to_csv(export_filename, index=False)

print(f"✅ Process Complete! File saved successfully as '{export_filename}'.")

🔄 Initializing Google Earth Engine...
✅ GEE Initialized Successfully.
📍 Study Area Loaded. Timeframe: 2017-2025
🚀 Extracting Monthly Constellation Data (S2 + L8 + L9)...
  [+] 2017-01 | Images: 02 | NBR: 0.309 | NDVI: 0.075
  [+] 2017-02 | Images: 02 | NBR: 0.383 | NDVI: 0.013
  [+] 2017-03 | Images: 03 | NBR: 0.352 | NDVI: 0.158
  [+] 2017-04 | Images: 02 | NBR: 0.221 | NDVI: 0.227
  [+] 2017-05 | Images: 04 | NBR: 0.218 | NDVI: 0.501
  [+] 2017-06 | Images: 05 | NBR: 0.164 | NDVI: 0.449
  [+] 2017-07 | Images: 06 | NBR: 0.098 | NDVI: 0.411
  [+] 2017-08 | Images: 07 | NBR: 0.069 | NDVI: 0.290
  [+] 2017-09 | Images: 05 | NBR: 0.088 | NDVI: 0.256
  [+] 2017-10 | Images: 04 | NBR: 0.037 | NDVI: 0.326
  [+] 2017-11 | Images: 04 | NBR: 0.047 | NDVI: 0.344
  [+] 2017-12 | Images: 01 | NBR: 0.214 | NDVI: 0.323
  [+] 2018-01 | Images: 02 | NBR: 0.374 | NDVI: 0.040
  [+] 2018-02 | Images: 02 | NBR: 0.304 | NDVI: 0.431
  [+] 2018-03 | Images: 01 | NBR: 0.347 | NDVI: 0.325
  [+] 2018-04 | Imag

# SCRIPT 02: Spatiotemporal Fire Frequency (Recurrence) Mapping
**Project:** Andean Fire Ecosystem Dynamics (2017-2025)

**Description:**
This script generates a pixel-wise Fire Frequency Map to visualize the spatiotemporal variability of fires. To avoid the visual clutter of overlapping burn perimeters over 9 years, it calculates the annual differenced Normalized Burn Ratio (dNBR) applying a validated threshold for high-Andean ecosystems (dNBR > 0.25).

These annual binary masks are summed to quantify exactly how many times each individual 30m pixel has burned over the study period, directly demonstrating the spatial patterns of recurrent burning.

In [ ]:
"""
================================================================================
SCRIPT 02: Spatiotemporal Fire Frequency (Recurrence) Mapping
Project: Andean Fire Ecosystem Dynamics (2017-2025)
Description: Generates a pixel-wise Fire Frequency Map by calculating annual
dNBR and accumulating binary burn masks (dNBR > 0.25) to quantify fire recurrence.
================================================================================
"""

import ee
import geemap
import time

# ------------------------------------------------------------------------------
# 1. INITIALIZATION
# ------------------------------------------------------------------------------
print("🔄 Initializing Google Earth Engine...")
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='my-project-123-477720') # Update with your project ID
print("✅ GEE Initialized Successfully.")

# ------------------------------------------------------------------------------
# 2. PARAMETERS AND ROI
# ------------------------------------------------------------------------------
# Replace with your actual asset path
ROI_PATH = 'projects/my-project-123-477720/assets/ACR_TITANKAYOCC_UNION'
roi = ee.FeatureCollection(ROI_PATH)

START_YEAR = 2017
END_YEAR = 2025
DNBR_THRESHOLD = 0.25  # Validated threshold for high-Andean ecosystems

print(f"📍 Study Area Loaded. Timeframe: {START_YEAR}-{END_YEAR}")

# ------------------------------------------------------------------------------
# 3. HELPER FUNCTIONS (HARMONIZATION)
# ------------------------------------------------------------------------------
def process_s2(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR').float()
    return image.updateMask(mask).addBands(nbr)

def process_landsat(image):
    qa = image.select('QA_PIXEL')
    cloudShadowBitMask = 1 << 4
    cloudsBitMask = 1 << 3
    mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0).And(qa.bitwiseAnd(cloudsBitMask).eq(0))
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    nir = opticalBands.select('SR_B5')
    swir = opticalBands.select('SR_B7')
    nbr = nir.subtract(swir).divide(nir.add(swir)).rename('NBR').float()
    return image.updateMask(mask).addBands(nbr)

def get_annual_nbr(year):
    """Generates an annual median NBR composite."""
    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = ee.Date.fromYMD(year, 12, 31)

    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(roi).filterDate(start_date, end_date).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 70)).map(process_s2).select('NBR')
    l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(roi).filterDate(start_date, end_date).filter(ee.Filter.lt('CLOUD_COVER', 70)).map(process_landsat).select('NBR')
    l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterBounds(roi).filterDate(start_date, end_date).filter(ee.Filter.lt('CLOUD_COVER', 70)).map(process_landsat).select('NBR')

    combined = s2.merge(l8).merge(l9)
    return combined.min().clip(roi)

# ------------------------------------------------------------------------------
# 4. FIRE FREQUENCY CALCULATION
# ------------------------------------------------------------------------------
print("🔥 Calculating annual dNBR and accumulating fire frequency...")

# Generate a list of annual NBR composites (starting one year prior for baseline)
annual_nbr_list = [get_annual_nbr(y) for y in range(START_YEAR - 1, END_YEAR + 1)]

# Create an empty image to accumulate fire counts
fire_frequency = ee.Image.constant(0).clip(roi).rename('Fire_Frequency')

# Calculate dNBR for each year (Pre-fire NBR - Post-fire NBR)
for i in range(len(annual_nbr_list) - 1):
    nbr_pre = annual_nbr_list[i]
    nbr_post = annual_nbr_list[i+1]

    # Calculate dNBR
    dnbr = nbr_pre.subtract(nbr_post).rename('dNBR')

    # Create binary mask for burned areas based on validated threshold
    burned_mask = dnbr.gt(DNBR_THRESHOLD)

    # Accumulate the fires
    fire_frequency = fire_frequency.add(burned_mask)

    current_year = START_YEAR + i
    print(f"  [+] Year {current_year} evaluated.")

# Mask out areas that never burned (Frequency == 0) to keep the map clean
fire_frequency = fire_frequency.updateMask(fire_frequency.gt(0))

# ------------------------------------------------------------------------------
# 5. EXPORT TO GOOGLE DRIVE
# ------------------------------------------------------------------------------
print("💾 Exporting Fire Frequency Map to Google Drive...")
export_task = ee.batch.Export.image.toDrive(
    image=fire_frequency.toInt(),
    description='Fire_Frequency_2017_2025',
    folder='Andean_Fire_Paper',
    fileNamePrefix='Fire_Frequency_Map',
    region=roi.geometry(),
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e10
)

export_task.start()

# Monitor the task
while export_task.active():
    print(f"  [⏳] Task State: {export_task.status()['state']}...")
    time.sleep(15)

print(f"✅ Process Complete! Task State: {export_task.status()['state']}")
print("Please check your Google Drive (folder: 'Andean_Fire_Paper') for the 'Fire_Frequency_Map.tif' file.")

🔄 Initializing Google Earth Engine...
✅ GEE Initialized Successfully.
📍 Study Area Loaded. Timeframe: 2017-2025
🔥 Calculating annual dNBR and accumulating fire frequency...
  [+] Year 2017 evaluated.
  [+] Year 2018 evaluated.
  [+] Year 2019 evaluated.
  [+] Year 2020 evaluated.
  [+] Year 2021 evaluated.
  [+] Year 2022 evaluated.
  [+] Year 2023 evaluated.
  [+] Year 2024 evaluated.
  [+] Year 2025 evaluated.
💾 Exporting Fire Frequency Map to Google Drive...
  [⏳] Task State: READY...
  [⏳] Task State: RUNNING...
  [⏳] Task State: RUNNING...
  [⏳] Task State: RUNNING...
  [⏳] Task State: RUNNING...
  [⏳] Task State: RUNNING...
  [⏳] Task State: RUNNING...
  [⏳] Task State: RUNNING...
  [⏳] Task State: RUNNING...
  [⏳] Task State: RUNNING...
✅ Process Complete! Task State: COMPLETED
Please check your Google Drive (folder: 'Andean_Fire_Paper') for the 'Fire_Frequency_Map.tif' file.


# SCRIPT 03: Climate Data Extraction (Precipitation) via CHIRPS
**Project:** Andean Fire Ecosystem Dynamics (2017-2025)

**Description:**
This script extracts monthly precipitation data from the CHIRPS dataset (Climate Hazards Group InfraRed Precipitation with Station data) to establish a robust hydrological baseline.

This dataset facilitates the subsequent computation of Time-Lag Correlations to evaluate Ecological Memory, allowing for the empirical demonstration of how high-Andean vegetation requires months of cumulative moisture to generate a greenness peak (NDVI), whereas structural combustion (NBR) is captured instantaneously.

In [ ]:
"""
================================================================================
SCRIPT 03: Climate Data Extraction (Precipitation) via CHIRPS
Project: Andean Fire Ecosystem Dynamics (2017-2025)
Description: Extracts monthly precipitation data from the CHIRPS dataset
to establish a hydrological baseline for time-lag correlation analysis.
================================================================================
"""

import ee
import geemap
import pandas as pd
import time

# ------------------------------------------------------------------------------
# 1. INITIALIZATION AND AUTHENTICATION
# ------------------------------------------------------------------------------
print("🔄 Initializing Google Earth Engine...")
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='my-project-123-477720') # Update with your project ID
print("✅ GEE Initialized Successfully.")

# ------------------------------------------------------------------------------
# 2. STUDY AREA & PARAMETERS
# ------------------------------------------------------------------------------
ROI_PATH = 'projects/my-project-123-477720/assets/ACR_TITANKAYOCC_UNION'
roi = ee.FeatureCollection(ROI_PATH)

START_YEAR = 2017
END_YEAR = 2025

print(f"📍 Study Area Loaded. Extracting climate data for {START_YEAR}-{END_YEAR}...")

# ------------------------------------------------------------------------------
# 3. EXTRACTION LOOP (CHIRPS DAILY -> MONTHLY AGGREGATION)
# ------------------------------------------------------------------------------
months = list(range(1, 13))
years = list(range(START_YEAR, END_YEAR + 1))
climate_data = []

print("🌧️ Extracting CHIRPS Precipitation Data...")

for year in years:
    for month in months:
        # Define exact start and end dates for the month
        start_date = f'{year}-{month:02d}-01'
        # advance 1 month to get the end date exclusively
        end_date = ee.Date(start_date).advance(1, 'month').format('YYYY-MM-dd').getInfo()

        # Load CHIRPS Daily dataset for the specific month
        chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                  .filterBounds(roi)
                  .filterDate(start_date, end_date))

        # Check if dataset has images for this month (e.g., handling recent months in 2025)
        if chirps.size().getInfo() > 0:
            # Sum all daily precipitation within the month
            monthly_precip = chirps.sum().clip(roi)

            # Reduce region to get the mean precipitation value over the entire ROI
            # CHIRPS native scale is ~5566 meters (0.05 degrees)
            stats = monthly_precip.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=roi.geometry(),
                scale=5566,
                maxPixels=1e9
            )

            precip_val = stats.get('precipitation').getInfo()

            if precip_val is not None:
                climate_data.append({
                    'Date': f'{year}-{month:02d}-15', # Mid-month format matching vegetation data
                    'Precipitation_mm': round(precip_val, 2)
                })
                print(f"  [+] {year}-{month:02d} | Monthly Rain: {precip_val:.2f} mm")
            else:
                print(f"  [-] {year}-{month:02d} | No valid data in ROI.")
        else:
            print(f"  [-] {year}-{month:02d} | CHIRPS data not yet available for this period.")

# ------------------------------------------------------------------------------
# 4. EXPORT TO CSV
# ------------------------------------------------------------------------------
print("💾 Saving climate data to CSV...")
df_clim = pd.DataFrame(climate_data)
df_clim['Date'] = pd.to_datetime(df_clim['Date'])
df_clim = df_clim.sort_values(by='Date')

export_filename = '4_Precipitation_Monthly.csv'
df_clim.to_csv(export_filename, index=False)

print(f"✅ Process Complete! File saved successfully as '{export_filename}'.")

🔄 Initializing Google Earth Engine...
✅ GEE Initialized Successfully.
📍 Study Area Loaded. Extracting climate data for 2017-2025...
🌧️ Extracting CHIRPS Precipitation Data...
  [+] 2017-01 | Monthly Rain: 172.71 mm
  [+] 2017-02 | Monthly Rain: 147.68 mm
  [+] 2017-03 | Monthly Rain: 165.45 mm
  [+] 2017-04 | Monthly Rain: 43.83 mm
  [+] 2017-05 | Monthly Rain: 33.83 mm
  [+] 2017-06 | Monthly Rain: 9.83 mm
  [+] 2017-07 | Monthly Rain: 12.23 mm
  [+] 2017-08 | Monthly Rain: 11.32 mm
  [+] 2017-09 | Monthly Rain: 46.34 mm
  [+] 2017-10 | Monthly Rain: 73.14 mm
  [+] 2017-11 | Monthly Rain: 84.69 mm
  [+] 2017-12 | Monthly Rain: 143.43 mm
  [+] 2018-01 | Monthly Rain: 193.36 mm
  [+] 2018-02 | Monthly Rain: 141.18 mm
  [+] 2018-03 | Monthly Rain: 151.31 mm
  [+] 2018-04 | Monthly Rain: 48.72 mm
  [+] 2018-05 | Monthly Rain: 26.63 mm
  [+] 2018-06 | Monthly Rain: 10.06 mm
  [+] 2018-07 | Monthly Rain: 21.31 mm
  [+] 2018-08 | Monthly Rain: 56.12 mm
  [+] 2018-09 | Monthly Rain: 56.25 mm


# SCRIPT 04: Spatiotemporal Trends, Global Impact & Zonal Statistics
**Project:** Andean Fire Ecosystem Dynamics (2017-2025)

**Description:**
To comprehensively evaluate the spatiotemporal variability of wildfires, this script extracts metrics at two simultaneous scales:
1. **Global Scale:** Calculates total burned hectares and mean severity (dNBR) across the entire ecosystem per year to establish general interannual trends.
2. **Zonal Scale:** Disaggregates the impact across specific management zones (e.g., Wilderness, Direct Use) to quantify localized vulnerability and relative ecological damage.

In [1]:
"""
================================================================================
SCRIPT 04: Spatiotemporal Trends, Global Impact & Zonal Statistics
Project: Andean Fire Ecosystem Dynamics (2017-2025)
Description: Extracts annual burned area and severity metrics at both ecosystem
(global) and management zone (zonal) scales.
================================================================================
"""

import ee
import pandas as pd

# ------------------------------------------------------------------------------
# 1. INITIALIZATION
# ------------------------------------------------------------------------------
print("🔄 Initializing Google Earth Engine...")
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='my-project-123-477720') # Update with your project ID
print("✅ GEE Initialized Successfully.")

# ------------------------------------------------------------------------------
# 2. STUDY AREA & PARAMETERS
# ------------------------------------------------------------------------------
# Asset for global metrics (Total Annual Impact)
UNION_PATH = 'projects/my-project-123-477720/assets/ACR_TITANKAYOCC_UNION'
roi_union = ee.FeatureCollection(UNION_PATH)

# Asset for internal subdivision statistics
ZONES_PATH = 'projects/my-project-123-477720/assets/ACR_TITANKAYOCC'
roi_zonas = ee.FeatureCollection(ZONES_PATH)

ZONE_PROPERTY = 'tz_cod' # Tu columna

START_YEAR = 2017
END_YEAR = 2025
DNBR_THRESHOLD = 0.25

print(f"📍 Study Areas Loaded. Timeframe: {START_YEAR}-{END_YEAR}")

# ------------------------------------------------------------------------------
# 3. HARMONIZATION FUNCTIONS
# ------------------------------------------------------------------------------
def process_s2(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI').float()
    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR').float()
    return image.updateMask(mask).addBands([ndvi, nbr])

def process_landsat(image):
    qa = image.select('QA_PIXEL')
    cloudShadowBitMask = 1 << 4
    cloudsBitMask = 1 << 3
    mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0).And(qa.bitwiseAnd(cloudsBitMask).eq(0))
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    red = opticalBands.select('SR_B4')
    nir = opticalBands.select('SR_B5')
    swir = opticalBands.select('SR_B7')
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI').float()
    nbr = nir.subtract(swir).divide(nir.add(swir)).rename('NBR').float()
    return image.updateMask(mask).addBands([ndvi, nbr])

def get_annual_composite(year):
    """Generates a cloud-free annual median composite."""
    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = ee.Date.fromYMD(year, 12, 31)

    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(roi_union).filterDate(start_date, end_date).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 70)).map(process_s2).select(['NDVI', 'NBR'])
    l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(roi_union).filterDate(start_date, end_date).filter(ee.Filter.lt('CLOUD_COVER', 70)).map(process_landsat).select(['NDVI', 'NBR'])
    l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterBounds(roi_union).filterDate(start_date, end_date).filter(ee.Filter.lt('CLOUD_COVER', 70)).map(process_landsat).select(['NDVI', 'NBR'])

    return s2.merge(l8).merge(l9).min().clip(roi_union)

# ------------------------------------------------------------------------------
# 4. EXTRACTION (GLOBAL IMPACT & ZONAL STATS)
# ------------------------------------------------------------------------------
print("📊 Extracting Multi-scale Statistics...")

ZONE_TRANSLATION = {
    'S': 'Wilderness Zone',
    'AD': 'Direct Use Zone',
    'UE': 'Special Use Zone',
    'REC': 'Recovery Zone',
    'T': 'Tourism and Recreation Zone'
}

annual_impact_data = []
zonal_data = []

# Baseline year for dNBR
comp_pre = get_annual_composite(START_YEAR - 1)

for year in range(START_YEAR, END_YEAR + 1):
    print(f"  [⏳] Processing Year {year}...")

    comp_post = get_annual_composite(year)

    # Calculate dNBR
    nbr_pre = comp_pre.select('NBR')
    nbr_post = comp_post.select('NBR')
    dnbr = nbr_pre.subtract(nbr_post).rename('dNBR')

    # Burned Area Image (Hectares)
    burned_area_img = dnbr.gt(DNBR_THRESHOLD).multiply(ee.Image.pixelArea()).divide(10000).rename('Burned_Ha')
    total_area_img = ee.Image.constant(1).multiply(ee.Image.pixelArea()).divide(10000).rename('Total_Zone_Ha')

    # ==========================================================================
    # A. GLOBAL ANNUAL IMPACT (Using roi_union)
    # ==========================================================================
    global_burned_stats = burned_area_img.reduceRegion(reducer=ee.Reducer.sum(), geometry=roi_union.geometry(), scale=30, maxPixels=1e13)
    global_dnbr_stats = dnbr.updateMask(dnbr.gt(DNBR_THRESHOLD)).reduceRegion(reducer=ee.Reducer.mean(), geometry=roi_union.geometry(), scale=30, maxPixels=1e13)

    gb_ha = global_burned_stats.get('Burned_Ha').getInfo()
    g_dnbr = global_dnbr_stats.get('dNBR').getInfo()

    annual_impact_data.append({
        'Year': year,
        'Burned_Area_ha': round(gb_ha, 2) if gb_ha else 0.0,
        'Mean_dNBR': round(g_dnbr, 4) if g_dnbr else 0.0
    })

    # ==========================================================================
    # B. ZONAL STATISTICS (Using roi_zonas)
    # ==========================================================================
    zones_list = roi_zonas.getInfo()['features']

    for feature in zones_list:
        zone_name_es = feature['properties'].get(ZONE_PROPERTY, 'Unknown_Zone')
        zone_name_es_clean = zone_name_es.strip().upper() if isinstance(zone_name_es, str) else zone_name_es
        zone_name_en = ZONE_TRANSLATION.get(zone_name_es_clean, zone_name_es)

        geom = ee.Geometry(feature['geometry'])

        burned_stats = burned_area_img.reduceRegion(reducer=ee.Reducer.sum(), geometry=geom, scale=30, maxPixels=1e9)
        total_stats = total_area_img.reduceRegion(reducer=ee.Reducer.sum(), geometry=geom, scale=30, maxPixels=1e9)
        indices_stats = comp_post.reduceRegion(reducer=ee.Reducer.mean(), geometry=geom, scale=30, maxPixels=1e9)

        b_ha = burned_stats.get('Burned_Ha').getInfo()
        t_ha = total_stats.get('Total_Zone_Ha').getInfo()
        n_mean = indices_stats.get('NBR').getInfo()
        nd_mean = indices_stats.get('NDVI').getInfo()

        zonal_data.append({
            'Year': year,
            'Zone': zone_name_en.title() if zone_name_en else 'Unknown Zone', # Sintaxis corregida
            'Burned_Ha': round(b_ha, 2) if b_ha else 0.0,
            'Total_Zone_Ha': round(t_ha, 2) if t_ha else 0.0,
            'NBR_Mean': round(n_mean, 4) if n_mean else None,
            'NDVI_Mean': round(nd_mean, 4) if nd_mean else None
        })

    # Advance to next year
    comp_pre = comp_post

# ------------------------------------------------------------------------------
# 5. EXPORT RESULTS
# ------------------------------------------------------------------------------
print("💾 Saving data to CSVs...")

# Export 2_Annual_Impact.csv
df_annual = pd.DataFrame(annual_impact_data)
df_annual.to_csv('2_Annual_Impact.csv', index=False)
print("✅ File 1: '2_Annual_Impact.csv' saved successfully.")

# Export 3_Zonal_Annual_Stats.csv
df_zonal = pd.DataFrame(zonal_data)
df_zonal = df_zonal.sort_values(by=['Zone', 'Year'])
df_zonal.to_csv('3_Zonal_Annual_Stats.csv', index=False)
print("✅ File 2: '3_Zonal_Annual_Stats.csv' saved successfully.")

try:
    from google.colab import files
    files.download('2_Annual_Impact.csv')
    files.download('3_Zonal_Annual_Stats.csv')
except Exception as e:
    print("Auto-download failed. Please download the files from the left panel in Colab.")

🔄 Initializing Google Earth Engine...
✅ GEE Initialized Successfully.
📍 Study Areas Loaded. Timeframe: 2017-2025
📊 Extracting Multi-scale Statistics...
  [⏳] Processing Year 2017...
  [⏳] Processing Year 2018...
  [⏳] Processing Year 2019...
  [⏳] Processing Year 2020...
  [⏳] Processing Year 2021...
  [⏳] Processing Year 2022...
  [⏳] Processing Year 2023...
  [⏳] Processing Year 2024...
  [⏳] Processing Year 2025...
💾 Saving data to CSVs...
✅ File 1: '2_Annual_Impact.csv' saved successfully.
✅ File 2: '3_Zonal_Annual_Stats.csv' saved successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# SCRIPT 05: Topographic Sampling for Environmental Drivers
**Project:** Andean Fire Ecosystem Dynamics (2017-2025)

**Description:**
This script performs a stratified spatial sampling to extract topographic variables (Elevation, Slope, Aspect) from the NASA SRTM 30m Digital Elevation Model. It reconstructs a consolidated binary mask of historically burned areas over the study period.

To ensure statistical balance for subsequent modeling, it extracts an equal number of burned and unburned spatial points (1000 each). This robust sampling strategy is critical for evaluating the environmental drivers of fire occurrence.

In [ ]:
"""
================================================================================
SCRIPT 05: Topographic Sampling for Environmental Drivers
Project: Andean Fire Ecosystem Dynamics (2017-2025)
Description: Extracts topographic variables using a stratified sampling approach
to compare burned vs. unburned areas.
================================================================================
"""

import ee
import pandas as pd

# ------------------------------------------------------------------------------
# 5. TOPOGRAPHIC SAMPLING FOR ENVIRONMENTAL DRIVERS (BURNED VS UNBURNED)
# ------------------------------------------------------------------------------
print("Starting Topographic Sampling (Elevation, Slope, Aspect)...")

# 1. Load Digital Elevation Model (NASA SRTM 30m)
dem = ee.Image('USGS/SRTMGL1_003').clip(roi_union)
elevation = dem.select('elevation').rename('Elevation')
slope = ee.Terrain.slope(dem).rename('Slope')
aspect = ee.Terrain.aspect(dem).rename('Aspect')

# 2. Re-create consolidated binary mask (1 = Burned ever, 0 = Never burned)
# Using the established function and threshold to maintain strict reproducibility
print("  [WAIT] Calculating pixel-by-pixel historical fire record...")
burned_any_year = ee.Image(0).clip(roi_union).rename('Burned_Class')
comp_pre_topo = get_annual_composite(START_YEAR - 1)

for year in range(START_YEAR, END_YEAR + 1):
    comp_post_topo = get_annual_composite(year)
    dnbr_topo = comp_pre_topo.select('NBR').subtract(comp_post_topo.select('NBR'))

    # If it exceeded the threshold that year, mark as burned
    is_burned = dnbr_topo.gt(DNBR_THRESHOLD)
    burned_any_year = burned_any_year.Or(is_burned)
    comp_pre_topo = comp_post_topo

# 3. Stack environmental variables with the fire class
topo_fire_img = ee.Image.cat([
    elevation,
    slope,
    aspect,
    burned_any_year.rename('Burned_Class')
])

# 4. Stratified Sampling
# Extracting exactly 1000 burned and 1000 unburned points for statistical balance
print("  [WAIT] Extracting 2000 spatial sampling points...")
samples = topo_fire_img.stratifiedSample(
    numPoints=1000,
    classBand='Burned_Class',
    region=roi_union.geometry(),
    scale=30,
    geometries=True # Important: saves X and Y coordinates
)

# 5. Process results and convert to Pandas DataFrame
sample_features = samples.getInfo()['features']

topo_data = []
for feat in sample_features:
    props = feat['properties']
    coords = feat['geometry']['coordinates']

    topo_data.append({
        'Burned_Class': 'Burned' if props.get('Burned_Class') == 1 else 'Unburned',
        'Elevation_m': round(props.get('Elevation', 0), 2),
        'Slope_deg': round(props.get('Slope', 0), 2),
        'Aspect_deg': round(props.get('Aspect', 0), 2),
        'Longitude': coords[0],
        'Latitude': coords[1]
    })

df_topo = pd.DataFrame(topo_data)

# Clean null values (e.g., points falling on polygon edges)
df_topo = df_topo.dropna()

# 6. Save and Download CSV
topo_csv = '5_Topographic_Samples.csv'
df_topo.to_csv(topo_csv, index=False)
print(f"Sampling complete. File '{topo_csv}' ready with {len(df_topo)} samples.")

try:
    from google.colab import files
    files.download(topo_csv)
except Exception as e:
    print("Auto-download failed. Please download from the left panel (folder icon).")